# Runtime curve walkthrough: hurricane wind × real estate

Reads the proposed model-v0.2 artifact and calls the curves through the cell's reference evaluator, the way a Hazard
M3 consumer would. Damage only: no frequency, no yearly loss. See
[`00_curve_curation_walkthrough.ipynb`](00_curve_curation_walkthrough.ipynb) for how the curves are built.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "docs/cells").is_dir() and (p / "scripts/reference_helpers").is_dir():
            return p
    raise RuntimeError("Run this notebook from inside the Damage_Modeling repository.")


ROOT = find_repo_root()
CELL_ID = "tropical_cyclone_wind_real_estate"
PROP = ROOT / "docs/cells" / CELL_ID / "proposed"
ARTIFACT_PATH = PROP / "tropical_cyclone_wind_real_estate__model_v0_2__docs_r1__curve_artifact.json"
REG = "tropical_cyclone_wind_real_estate__model_v0_1__docs_r1"  # tag of the cell registers (source, claims, tiers, decisions)
OUT = ROOT / "notebooks/tropical_cyclone_wind/real_estate/outputs"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts/reference_helpers"))
import tropical_cyclone_wind_real_estate_curve_eval as ev  # the cell's reference evaluator

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

ART = json.loads(ARTIFACT_PATH.read_text())
# Replacement cost new from the Hazard reference profiles r1 (Q3 2025 Dallas prices). Illustration only:
# the artifacts withhold scenario dollars until the consumer qualifies each building value.
BUDGETS_USD = {"RE_RES_SFD_1S_SLAB": 405_000, "RE_IND_WAREHOUSE": 11_000_000, "RE_COM_OFFICE_3S_SLAB": 10_800_000}
NAMES = {"RE_RES_SFD_1S_SLAB": "House", "RE_IND_WAREHOUSE": "Warehouse", "RE_COM_OFFICE_3S_SLAB": "Office",
         "RE_COM_LOWRISE": "Retail (unbound)"}


def table(name: str) -> pd.DataFrame:
    return pd.read_csv(PROP / name)


def save(fig, name: str) -> None:
    fig.savefig(OUT / name, dpi=110, bbox_inches="tight", metadata={"Software": None})


def decision_log(path: Path) -> pd.DataFrame:
    """Decision-log rows: ID, whether JL decided it, and the decision text (shortened)."""
    rows = []
    for line in path.read_text().splitlines():
        if not line.startswith("| D-"):
            continue
        cells = [c.strip() for c in line.strip().strip("|").split(" | ")]
        text = re.sub(r"\*\*", "", cells[1])
        rows.append({"id": cells[0], "status": "JL decided" if "JL decided" in cells[1] else "proposed",
                      "decision": text if len(text) <= 240 else text[:240] + " …"})
    return pd.DataFrame(rows)


print("artifact:", ARTIFACT_PATH.relative_to(ROOT))
print(ART["cell_id"], "|", ART["semantic_damage_model_version"], "|", ART["damage_code_id"],
      "| promotion:", ART["promotion_status"], "| canonical:", ART["canonical_runtime_artifact"])


def call(request: dict) -> dict:
    """Send one request through the cell's reference evaluator; return a flat row."""
    try:
        out = ev.evaluate_damage_call(ART, request)
    except ev.DamageCallError as err:
        return {"status": "error", "curve_id": None, "damage_ratio": None, "reason": err.code}
    r = out["failure_unit_results"][0]
    return {"status": r["status"], "curve_id": r["curve_id"], "damage_ratio": r["scalar_central_dr"],
            "reason": "; ".join(r["withheld_reason_codes"]) or None}

TERRAINS = ["open", "light_suburban", "suburban", "light_trees", "trees"]

artifact: docs/cells/tropical_cyclone_wind_real_estate/proposed/tropical_cyclone_wind_real_estate__model_v0_2__docs_r1__curve_artifact.json
tropical_cyclone_wind_real_estate | model v0.2 | TROPICAL_CYCLONE_WIND_REAL_ESTATE_HAZUS_SCREENING_V0_2 | promotion: proposed | canonical: False


## 1. What the artifact contains

In [2]:
identity = pd.DataFrame([
    {"field": "cell", "value": ART["cell_id"]},
    {"field": "damage code", "value": ART["damage_code_id"]},
    {"field": "model / docs", "value": f'{ART["semantic_damage_model_version"]} / {ART["documentation_revision"]}'},
    {"field": "lifecycle / promotion", "value": f'{ART["lifecycle_state"]} / {ART["promotion_status"]}'},
    {"field": "canonical runtime artifact", "value": ART["canonical_runtime_artifact"]},
    {"field": "model grade", "value": ART["model_grade"]},
])
display(identity)
units = pd.DataFrame([{k: u.get(k) for k in ("id", "treatment", "denominator")} for u in ART["failure_units"]])
display(units)
pathways = pd.DataFrame([{
    "pathway": p["pathway_id"],
    "axis field": p["hazard_axis"]["preferred_input_field"],
    "unit": p["hazard_axis"]["unit"],
    "valid range": p["hazard_axis"]["valid_range"],
    "selectors": ", ".join(s["field"] for s in p["selector_logic"] if s.get("required")),
    "conditioners": ", ".join(c["field"] for c in p["conditioner_logic"]) or "-",
    "records": len(p["curve_records"]),
} for p in ART["pathways"]])
display(pathways)

,field,value
0,cell,tropical_cyclone_wind_real_estate
1,damage code,TROPICAL_CYCLONE_WIND_REAL_ESTATE_HAZUS_SCREENING_V0_2
2,model / docs,model v0.2 / docs r1
3,lifecycle / promotion,scaffold / proposed
4,canonical runtime artifact,False
5,model grade,screening_source_native_adopted_tables


,id,treatment,denominator
0,TCRE_BUILDING_STRUCTURE,primary_nonzero,"structure replacement cost new of the same reference building (no land, contents, site works or depreciation)"
1,TCRE_BUILDING_CONTENTS,withheld,contents value (deferred)


,pathway,axis field,unit,valid range,selectors,conditioners,records
0,tropical_cyclone_wind,tc_peak_gust_3s_10m_mph,mph,"[50.0, 250.0]","reference_building_id, hazus_terrain_class",-,20


## 2. Request contract

Building and terrain class are required; wind characteristics are optional and only the reference value of each is published.

In [3]:
display(pd.DataFrame(ART["evaluation_contract"]["accepted_axis_payloads"]))
sel = pd.DataFrame(ART["pathways"][0]["selector_logic"])[["field", "required", "allowed"]]
display(sel)

,mode,required_fields
0,open_terrain_gust_mph,[tc_peak_gust_3s_10m_mph]


,field,required,allowed
0,reference_building_id,True,"[RE_RES_SFD_1S_SLAB, RE_COM_LOWRISE, RE_IND_WAREHOUSE, RE_COM_OFFICE_3S_SLAB]"
1,hazus_terrain_class,True,"[open, light_suburban, suburban, light_trees, trees]"
2,hazus_garage_door,False,[none]
3,hazus_metal_deck_attachment,False,[standard]
4,hazus_roof_cover,False,"[shingle, bur, spm]"
5,hazus_roof_deck_age,False,[new_or_average]
6,hazus_roof_deck_attachment,False,[6d_6_12]
7,hazus_roof_shape,False,[gable]
8,hazus_roof_to_wall_connection,False,[toe_nail]
9,hazus_secondary_water_resistance,False,[no]


## 3. Calling the curves

Each profile building in open and suburban terrain.

In [4]:
rows = []
for b in ("RE_RES_SFD_1S_SLAB", "RE_IND_WAREHOUSE", "RE_COM_OFFICE_3S_SLAB"):
    for t in ("open", "suburban"):
        for g in (90, 100, 110, 120, 130, 140, 160):
            res = call({"pathway_id": "tropical_cyclone_wind", "reference_building_id": b, "hazus_terrain_class": t,
                        "tc_peak_gust_3s_10m_mph": float(g)})
            rows.append({"building": NAMES[b], "terrain": t, "gust_mph": g, "damage_ratio": res["damage_ratio"]})
grid = pd.DataFrame(rows).pivot_table(index=["building", "terrain"], columns="gust_mph", values="damage_ratio")
display(grid.style.format("{:.1%}"))

## 4. Fail-closed behaviour

In [5]:
base = {"pathway_id": "tropical_cyclone_wind", "reference_building_id": "RE_RES_SFD_1S_SLAB",
        "hazus_terrain_class": "suburban", "tc_peak_gust_3s_10m_mph": 130.0}
cases = {
    "valid request": base,
    "below 50 mph (source zero)": {**base, "tc_peak_gust_3s_10m_mph": 40.0},
    "above 250 mph": {**base, "tc_peak_gust_3s_10m_mph": 260.0},
    "terrain missing": {k: v for k, v in base.items() if k != "hazus_terrain_class"},
    "terrain as a roughness number": {**base, "hazus_terrain_class": 0.35},
    "unpublished variant (shutters)": {**base, "hazus_shutters": "yes"},
    "characteristic the house does not have": {**base, "hazus_window_area": "low"},
    "contents": {**base, "failure_unit_id": "TCRE_BUILDING_CONTENTS"},
}
display(pd.DataFrame([{"case": k, **call(v)} for k, v in cases.items()]))

,case,status,curve_id,damage_ratio,reason
0,valid request,conditional,TCRE_RES_HAZUS_WSF1_SUBURBAN,0.263409,NaN
1,below 50 mph (source zero),conditional,TCRE_RES_HAZUS_WSF1_SUBURBAN,0.000000,NaN
2,above 250 mph,withheld,TCRE_RES_HAZUS_WSF1_SUBURBAN,NaN,ABOVE_SOURCE_VALID_RANGE
3,terrain missing,error,NaN,NaN,TERRAIN_CLASS_REQUIRED
4,terrain as a roughness number,error,NaN,NaN,TERRAIN_CLASS_UNKNOWN
5,unpublished variant (shutters),withheld,NaN,NaN,WIND_CHARACTERISTIC_VARIANT_NOT_PUBLISHED
6,characteristic the house does not have,error,NaN,NaN,WIND_CHARACTERISTIC_NOT_APPLICABLE
7,contents,withheld,NaN,NaN,NO_RUNTIME_CURVE_FOR_PATHWAY_UNIT; CONTENTS_DEFERRED_SEPARATE_FAILURE_UNIT


## 5. From damage ratio to dollars (illustration)

The Hazard profile budgets are used only to show `loss = DR × replacement cost new`; the artifact withholds dollars until Hazard qualifies each value.

In [6]:
rows = []
for b, budget in BUDGETS_USD.items():
    res = call({"pathway_id": "tropical_cyclone_wind", "reference_building_id": b, "hazus_terrain_class": "suburban",
                "tc_peak_gust_3s_10m_mph": 120.0})
    rows.append({"building": NAMES[b], "replacement cost new": budget, "DR at 120 mph, suburban": res["damage_ratio"],
                 "loss (illustration)": res["damage_ratio"] * budget})
display(pd.DataFrame(rows).style.format({"replacement cost new": "${:,.0f}", "DR at 120 mph, suburban": "{:.1%}",
                                         "loss (illustration)": "${:,.0f}"}))

,building,replacement cost new,"DR at 120 mph, suburban",loss (illustration)
0,House,"$405,000",9.3%,"$37,463"
1,Warehouse,"$11,000,000",5.2%,"$570,940"
2,Office,"$10,800,000",10.1%,"$1,090,390"


## 6. Capability

In [7]:
cap = ART["capability_declaration"]
rows = []
for pc in cap["pathway_capabilities"]:
    rows.append({"pathway": pc["pathway_id"], "failure-unit DR": pc["failure_unit_scalar_dr"],
                 "scenario loss": pc["scenario_loss_given_value_basis"], "spread": pc["curve_intrinsic_spread"],
                 "withheld units": ", ".join(w["failure_unit_id"] for w in pc.get("withheld_failure_units", []))})
display(pd.DataFrame(rows))
print("Annual and tail metrics:", cap["consumer_annual_metrics"]["status_before_promotion"],
      "(owner:", cap["consumer_annual_metrics"]["computation_owner"] + ")")
print("Promotion gate:", cap["promotion_gate"]["status"])
for item in cap["promotion_gate"]["required_before_canonical_use"]:
    print("  -", item)

,pathway,failure-unit DR,scenario loss,spread,withheld units
0,tropical_cyclone_wind,conditional,withheld,not_carried,TCRE_BUILDING_CONTENTS


Annual and tail metrics: withheld_noncanonical_proposal (owner: downstream_consumer)
Promotion gate: blocked
  - owner accepts scope D-TCRE-01 to D-TCRE-05 and D-TCRE-08 to D-TCRE-13, and selections D-TCRE-17 to D-TCRE-25
  - consumer value qualification of each reference budget (shared with flood_real_estate)
  - Hazard supplies a terrain class per building location
  - independent review of the package


## 7. Not in this notebook

- Frequency and yearly loss belong to Hazard.
- Re-extracting the Hazus rows needs the FEMA Hazus 6.1 wind extract and SimCenter's Hazus 5.1 copy:
  `scripts/reference_helpers/extract_tropical_cyclone_wind_real_estate_v0_2_hazus_ordinates.py`.
- The Hazard loader probe is `check_tropical_cyclone_wind_real_estate_hazard_loader.py`.